# Notebook 4 — Kibana Exploration

*Hands-on time: ~25 minutes*

In this final notebook, we step into **Kibana** — the visual UI for ElasticSearch.
Most of this notebook is a guided walkthrough with screenshots and instructions;
the code cells prepare the data and verify your Kibana setup.

You will:
1. Access Kibana and navigate the interface
2. Use Dev Tools Console to run queries
3. Create a Data View for our index
4. Explore data in Discover
5. Build visualizations with Lens
6. Assemble a Dashboard

**Prerequisites:** Notebooks 1–3 completed, Docker services running.

---
## 0. Verify Kibana is Running

In [1]:
import urllib.request
import json

KIBANA_URL = "http://localhost:5601"

try:
    req = urllib.request.Request(f"{KIBANA_URL}/api/status")
    with urllib.request.urlopen(req, timeout=10) as resp:
        status = json.loads(resp.read())
    print(f"Kibana status: {status['status']['overall']['level']}")
    print(f"Version:       {status['version']['number']}")
    print(f"\n✅ Kibana is running at {KIBANA_URL}")
except Exception as e:
    print(f"❌ Cannot reach Kibana at {KIBANA_URL}: {e}")
    print("Make sure Docker services are running: docker compose up -d")

Kibana status: available
Version:       9.3.0

✅ Kibana is running at http://localhost:5601


In [2]:
# Verify our index has data
from elasticsearch import Elasticsearch

client = Elasticsearch("http://localhost:9200")
INDEX_NAME = "neuroimaging"

count = client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' has {count['count']} documents")
assert count['count'] > 0, "No data! Run Notebook 1 first."

Index 'neuroimaging' has 4423 documents


---
## 1. Navigate to Kibana

Open your browser and go to: **http://localhost:5601**

### First-time setup
1. Kibana may show a Welcome page — click **"Explore on my own"**
2. If prompted about sample data, skip it — we have our own data

### The sidebar

Kibana 9.x organises navigation into **solution-based sections**. You'll see
these top-level items in the left sidebar:

| Section | Key items for this course |
| --- | --- |
| **Analytics** | Discover, Dashboard, Visualize Library |
| **Elasticsearch** | Indices, Connectors |
| **Management** | Dev Tools (Console), Stack Management (Data Views) |

> Observability and Security also appear but are not used in this course.
>
> You can collapse the sidebar with the **«** button at the bottom to get more
> workspace area — hover over the collapsed strip to expand it temporarily.

---
## 2. Dev Tools Console

The Dev Tools Console is the fastest way to interact with ES from Kibana.

### How to access
1. In the left sidebar, expand **Management** → click **Dev Tools**
2. Or navigate directly to: **http://localhost:5601/app/dev_tools#/console**

### Try these queries in the Console

```json
// Check cluster health
GET _cluster/health

// Count documents in our index
GET neuroimaging/_count

// View the index mapping
GET neuroimaging/_mapping

// How many scans per dataset?
GET neuroimaging/_search
{
  "size": 0,
  "aggs": {
    "by_dataset": {
      "terms": { "field": "dataset" }
    }
  }
}

// Filter to a single dataset
GET neuroimaging/_search
{
  "query": {
    "term": { "dataset": "ds000117" }
  },
  "size": 3,
  "_source": {
    "excludes": ["metadata_embedding"]
  }
}

// Scan types across all datasets
GET neuroimaging/_search
{
  "size": 0,
  "aggs": {
    "by_suffix": {
      "terms": { "field": "suffix" }
    }
  }
}
```

> **Tip:** Place your cursor on a query and press **Ctrl+Enter** (or click the ▶ button) to execute it.

---
## 3. Create a Data View

A **Data View** (formerly *Index Pattern*) tells Kibana which ES index to
visualize.

### Option A — from Stack Management
1. In the sidebar: **Management** → **Stack Management** → **Data Views** (under the *Kibana* heading)
2. Click **"Create data view"**
3. Settings:
   - **Name**: `neuroimaging`
   - **Index pattern**: `neuroimaging`
   - **Timestamp field**: select *"I don't want to use the time filter"* (our data has no timestamps)
4. Click **Save data view to Kibana**

### Option B — inline from Discover or Lens
1. Open **Analytics** → **Discover**
2. Click the data view dropdown at the top of the page
3. Click **"Create a data view"** and fill in the same settings as above

You should see all your fields listed with their types (keyword, float,
dense_vector, etc.).

---
## 4. Discover — Explore Raw Data

### Steps
1. In the sidebar: **Analytics** → **Discover**
2. Select the **neuroimaging** data view from the dropdown (top of the page)
3. You should see several thousand documents listed (one per NIfTI file across all datasets)

### Try these explorations

**Add columns:** Click the **+** button next to field names in the left sidebar:
- `dataset`, `subject`, `suffix`, `task`, `Manufacturer`

**Filter:** Click **+ Add filter** at the top:
- Field: `dataset`, Operator: `is`, Value: `ds000117` → Apply
- This shows only scans from the ds000117 multi-modal dataset

**Search bar (KQL):** Type in the search bar at the top:
- `dataset: "ds000117" AND suffix: bold`
- `MagneticFieldStrength >= 3`
- `Manufacturer: "Siemens" AND suffix: dwi`

**Inspect a document:** Click the expand arrow (▶) on any row to see all
fields, including metadata like `RepetitionTime`, `FlipAngle`, and
`InstitutionName`.

---
## 5. Build Visualizations with Lens

Lens is Kibana's drag-and-drop visualization builder. In 9.x, the recommended
workflow is to create visualizations **from inside a Dashboard**.

### Get started
1. **Analytics** → **Dashboard** → **Create dashboard**
2. Click **Add** (top toolbar) — this opens the Lens editor directly
3. Make sure the **neuroimaging** data view is selected (dropdown at the top)

> After building each visualization below, click **Save and return** to add it
> to the dashboard, then click **Add** again for the next one.

---

### Visualization 1: Pie Chart — Scans by Dataset

1. In the Lens editor, change the chart type dropdown to **Pie**
2. Drag `dataset` from the field list into the **Slice by** drop zone in the
   layer pane
3. The metric auto-sets to **Count** — you'll see many slices representing the
   various bids-examples datasets (the largest are `7t_trt`, `ds000117`, etc.)
4. Click **Save and return**

---

### Visualization 2: Stacked Bar — Scan Types per Dataset

1. Click **Add** to open a new Lens editor
2. Chart type: **Bar vertical stacked**
3. Drag `dataset` to the **Horizontal axis**
4. Drag `suffix` to the **Break down by** area
5. Vertical axis stays as **Count** (default)
6. You'll see how each dataset contributes different scan types — `bold` scans
   dominate overall, but many datasets also contribute `T1w`, `dwi`, `FLAIR`,
   `FLASH`, and other modalities
7. Click **Save and return**

---

### Visualization 3: Histogram — MagneticFieldStrength Distribution

1. Click **Add** → new Lens editor
2. Chart type: **Bar vertical**
3. Drag `MagneticFieldStrength` to the **Horizontal axis** — Lens
   auto-configures it as an interval histogram
4. Vertical axis: **Count**
5. You should see bars at **3.0 T** (dominant), **1.5 T**, and possibly **7 T**
   from the 7t_trt dataset
6. Click **Save and return**

---

### Visualization 4: Data Table — Dataset Overview

1. Click **Add** → new Lens editor
2. Chart type: **Table**
3. Drag `dataset` to the **Rows** area
4. The first metric column is already **Count**
5. Click **Add** (or **+**) next to Metrics → choose **Unique count** of
   `subject`
6. Add another metric → **Unique count** of `suffix`
7. You now have a compact summary: scan count, number of subjects, and number
   of scan types per dataset
8. Click **Save and return**

---
## 6. Arrange and Save the Dashboard

After creating all four visualizations you should be back on the dashboard
canvas with four panels.

1. **Resize and arrange** the panels by dragging their edges:
   - Top-left: Pie chart (Scans by Dataset)
   - Top-right: Stacked bar (Scan Types per Dataset)
   - Bottom-left: Histogram (MagneticFieldStrength)
   - Bottom-right: Data table (Dataset Overview)
2. **Try cross-filtering:** click the **ds000117** slice in the pie chart — all
   other panels instantly filter to show only that dataset's data. Click again
   to clear the filter.
3. **Save** the dashboard as **"Neuroimaging Overview"**

> **Key insight:** Clicking any element in any visualization filters ALL other
> panels on the dashboard. This linked filtering is one of Kibana's most
> powerful features for exploratory data analysis.

---
## 7. Bonus: kNN Queries in Dev Tools

You can run vector search directly from the Kibana Dev Tools Console.
The cell below generates a query vector you can copy-paste into Kibana.

In [3]:
from sentence_transformers import SentenceTransformer
import json

model = SentenceTransformer("all-MiniLM-L6-v2")

query_text = "Siemens scanner diffusion brain imaging"
query_vec = model.encode(query_text).tolist()

# Build the kNN query for copy-paste into Kibana Dev Tools
kibana_query = {
    "knn": {
        "field": "metadata_embedding",
        "query_vector": query_vec,
        "k": 5,
        "num_candidates": 50
    },
    "_source": {
        "excludes": ["metadata_embedding"]
    }
}

print("Copy this into Kibana Dev Tools Console:")
print("=" * 50)
print(f"GET neuroimaging/_search")
print(json.dumps(kibana_query, indent=2))
print()
print(f"Query text: '{query_text}'")
print("The results should span multiple datasets (ds000117, eeg_rest_fmri, genetics_ukbb).")

/home/scil/dev/neuropoly-db/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Copy this into Kibana Dev Tools Console:
GET neuroimaging/_search
{
  "knn": {
    "field": "metadata_embedding",
    "query_vector": [
      0.0370308980345726,
      -0.05289691686630249,
      -0.016984565183520317,
      -0.02579663321375847,
      0.010325209237635136,
      -0.07908934354782104,
      0.02607683092355728,
      -0.030423641204833984,
      0.0004037587495986372,
      -0.02835622988641262,
      0.03073304332792759,
      0.01867862045764923,
      -0.02558950148522854,
      0.0438879057765007,
      -0.09011928737163544,
      -0.12318161875009537,
      -0.048399314284324646,
      -0.051280323415994644,
      0.00510068004950881,
      0.030745700001716614,
      -0.10698248445987701,
      -0.06924790143966675,
      -0.10032277554273605,
      -0.10050052404403687,
      0.020322483032941818,
      0.10742256045341492,
      0.023284077644348145,
      -0.11289609968662262,
      0.016303785145282745,
      -0.04941842705011368,
      0.08862218260765076,
 

### Try it
1. In the sidebar: **Management** → **Dev Tools**
2. Paste the query above
3. Execute with **Ctrl+Enter**
4. Inspect the results — you should see hits from multiple datasets (the query
   mentions "Siemens" and "diffusion", so expect dwi scans from datasets that
   include diffusion imaging such as ds000117)

> This is the production workflow: generate embedding vectors in your
> application code, then send kNN queries to ES.

---
## Verification Checklist

Before finishing, verify you've completed these steps:

In [5]:
checklist = [
    "Accessed Kibana at http://localhost:5601",
    "Ran a query in Dev Tools Console (Management → Dev Tools)",
    "Created a Data View for 'neuroimaging'",
    "Explored documents in Discover with filters and KQL",
    "Built a Pie chart (Scans by Dataset)",
    "Built a Stacked Bar chart (Scan Types per Dataset)",
    "Built a Histogram (MagneticFieldStrength Distribution)",
    "Built a Data Table (Dataset Overview)",
    "Arranged all four panels on a Dashboard",
    "Tested linked cross-filtering (click a pie slice)",
    "Ran a kNN query in Dev Tools Console",
]

print("🏁 Course Completion Checklist")
print("=" * 40)
for i, item in enumerate(checklist, 1):
    print(f"  [x] {i:2d}. {item}")
print(f"\nTotal items: {len(checklist)}")

🏁 Course Completion Checklist
  [x]  1. Accessed Kibana at http://localhost:5601
  [x]  2. Ran a query in Dev Tools Console (Management → Dev Tools)
  [x]  3. Created a Data View for 'neuroimaging'
  [x]  4. Explored documents in Discover with filters and KQL
  [x]  5. Built a Pie chart (Scans by Dataset)
  [x]  6. Built a Stacked Bar chart (Scan Types per Dataset)
  [x]  7. Built a Histogram (MagneticFieldStrength Distribution)
  [x]  8. Built a Data Table (Dataset Overview)
  [x]  9. Arranged all four panels on a Dashboard
  [x] 10. Tested linked cross-filtering (click a pie slice)
  [x] 11. Ran a kNN query in Dev Tools Console

Total items: 11


---
## Summary — Course Complete!

Congratulations! Across all 4 notebooks you have:

| Notebook | What you learned |
| --- | --- |
| **NB1** | BIDS exploration with pybids, ES index mapping, embedding generation, bulk ingestion |
| **NB2** | BM25 full-text search, term/range filters, bool compound queries, aggregations |
| **NB3** | kNN vector search, filtered kNN, hybrid search, parameter tuning |
| **NB4** | Kibana 9.x navigation, Dev Tools Console, Discover, Lens visualizations, cross-dataset Dashboards |

### What you've built
- A **vector-enabled search engine** for neuroimaging metadata across **all bids-examples datasets** (~76 datasets, 4,000+ scans)
- That supports **keyword**, **semantic**, and **hybrid** search
- With a **Kibana dashboard** for visual cross-dataset exploration
- All running locally in **Docker**

See `docs/05-next-steps.md` for ideas on scaling this to production.